In [ ]:
%%writefile ambiente.h
// Contiene: i parametri, gli ostacoli (scatole, sfere, fori), la maschera della griglia,
// le sorgenti della passata 1 (bordo degli ostacoli)
#pragma once           // il file viene incluso una sola volta
#include <vector>
#include <cmath>
#include <algorithm>

// Forme con cui si descrive l'ambiente (coordinate nel cubo [-1, 1]^3)
struct Punto   { double x, y, z; };
struct Scatola { double x0, x1, y0, y1, z0, z1; };       // parallelepipedo: x0 <= x <= x1, y0 <= y <= y1, z0 <= z <= z1
struct Sfera   { double cx, cy, cz, r; };                 // centro (cx, cy, cz) e raggio r


// AMBIENTE: per cambiarlo si modifica solo questa parte
// =====================================================================
// Stanza = cubo [-1, 1]^3, z verticale (z = -1 pavimento). Fuori dal cubo = parete (INF).
const double MARGINE = 0.06;                             // distanza minima dagli ostacoli (raggio del drone + sicurezza)
const double D_SICUREZZA = 0.3;                          // distanza dagli ostacoli oltre la quale si va a velocita' piena
const double V_MIN = 0.1;                                // velocita' minima, a ridosso degli ostacoli (> 0)

const Punto A = { -0.8, -0.6, -0.6 };                    // partenza del drone: prima meta' della stanza, in basso
const Punto B = {  0.85, -0.4, -0.7 };                   // arrivo (sorgente del FIM): dietro l'armadio

const std::vector<Scatola> SCATOLE = {                   // {x0, x1, y0, y1, z0, z1}
    { -0.05, 0.05, -1.0,  1.0, -1.0,  1.0 },             // muro a x = 0: divide la stanza in due
    {  0.35, 0.65, -0.7, -0.1, -1.0, -0.2 },             // armadio appoggiato a terra, oltre il muro
};
const std::vector<Sfera> SFERE = {};                     // {cx, cy, cz, r}: nessuna sfera, per ora
const std::vector<Scatola> FORI = {                      // zone in cui l'ostacolo viene tolto
    { -0.05, 0.05, -0.3,  0.3, -1.0,  0.0 },             // porta nel muro: in basso, dal pavimento a meta' altezza
    { -0.05, 0.05,  0.5,  0.8,  0.4,  0.8 },             // finestra nel muro: in alto
};
// =====================================================================

// Vero se il punto (x, y, z) e' dentro la scatola s (bordi compresi)
inline bool dentro(const Scatola& s, double x, double y, double z)
{
    return s.x0 <= x && x <= s.x1 && s.y0 <= y && y <= s.y1 && s.z0 <= z && z <= s.z1;
}

// Vero se il punto (x, y, z) e' dentro la sfera s (distanza dal centro <= raggio)
inline bool dentro(const Sfera& s, double x, double y, double z)
{
    return (x - s.cx) * (x - s.cx) + (y - s.cy) * (y - s.cy) + (z - s.cz) * (z - s.cz) <= s.r * s.r;
}

// Vero se il punto (x, y, z) e' dentro un ostacolo (i FORI tolgono l'ostacolo)
inline bool ostacolo(double x, double y, double z)
{
    for (const auto& f : FORI)    if (dentro(f, x, y, z)) return false;
    for (const auto& s : SCATOLE) if (dentro(s, x, y, z)) return true;
    for (const auto& s : SFERE)   if (dentro(s, x, y, z)) return true;
    return false;
}

// Maschera della griglia: libero = 1, ostacolo = 0 (resta INF)
inline std::vector<unsigned char> costruisci_maschera(int N)
{
    const double h = 2.0 / (N - 1);
    const int n = N * N * N;
    std::vector<unsigned char> mask(n);
    for (int q = 0; q < n; ++q) {                     // un nodo alla volta
    const int j = q % N, i = (q / N) % N, k = q / (N * N);
    mask[q] = ostacolo(-1.0 + j * h, -1.0 + i * h, -1.0 + k * h) ? 0 : 1;
    }
    // Margine: diventa ostacolo ogni nodo entro MARGINE da un ostacolo
    const int r = (int)std::ceil(MARGINE / h);               // raggio sfera intorno all'ostacolo
    std::vector<unsigned char> margin_mask = mask;
    for (int q = 0; q < n; ++q) {
        if (mask[q]) continue;                                   
    const int k = q / (N * N), i = (q / N) % N, j = q % N;
    const bool bordo = (j > 0 && mask[q - 1]) || (j < N - 1 && mask[q + 1]) ||          // che toccano un nodo libero
                       (i > 0 && mask[q - N]) || (i < N - 1 && mask[q + N]) ||
                       (k > 0 && mask[q - N * N]) || (k < N - 1 && mask[q + N * N]);
    if (!bordo) continue;
    for (int dk = -r; dk <= r; ++dk)                          // tutti i nodi entro la sferadi raggio MARGINE
        for (int di = -r; di <= r; ++di)
            for (int dj = -r; dj <= r; ++dj) {
                const int kk = k + dk, ii = i + di, jj = j + dj;
                if (kk < 0 || kk >= N || ii < 0 || ii >= N || jj < 0 || jj >= N) continue;
                if ((dk * dk + di * di + dj * dj) * h * h <= MARGINE * MARGINE)
                    margin_mask[(kk * N + ii) * N + jj] = 0;
            }
}
return margin_mask;
}
// Sorgenti della passata 1: i nodi liberi che toccano un ostacolo
inline std::vector<int> nodi_di_bordo(const std::vector<unsigned char>& mask, int N)
{
    std::vector<int> bordo;
    for (int k = 0; k < N; ++k)
        for (int i = 0; i < N; ++i)
            for (int j = 0; j < N; ++j) {
                const int q = (k * N + i) * N + j;
                if (!mask[q]) continue;                                  // solo nodi liberi...
                if ((j > 0 && !mask[q - 1]) || (j < N - 1 && !mask[q + 1]) ||
                    (i > 0 && !mask[q - N]) || (i < N - 1 && !mask[q + N]) ||
                    (k > 0 && !mask[q - N * N]) || (k < N - 1 && !mask[q + N * N]))
                    bordo.push_back(q);                                  // ...che hanno un vicino ostacolo
            }
    return bordo;
}

// FM2: dalla distanza d dagli ostacoli alla lentezza F = 1/v della passata 2
// v = V_MIN + (1 - V_MIN) * min(d / D_SICUREZZA, 1): F = 10 sul bordo, 1 lontano
inline std::vector<float> lentezza_fm2(const std::vector<float>& d)
{
    std::vector<float> F(d.size());
    for (size_t q = 0; q < d.size(); ++q) {
        const double r = std::min((double)d[q] / D_SICUREZZA, 1.0);       // distanza normalizzata: 0 sul bordo, 1 oltre D_SICUREZZA (o se d = INF)
        F[q] = (float)(1.0 / (V_MIN + (1.0 - V_MIN) * r));
    }
    return F;
}
// Controllo iniziale: A e B non devono stare dentro un ostacolo
inline bool ambiente_valido()
{
    return !ostacolo(A.x, A.y, A.z) && !ostacolo(B.x, B.y, B.z);
}

// Dal punto (x, y, z) all'indice del nodo piu' vicino (serve per A e B)
int nodo(int N, double x, double y, double z){
    const double h = 2.0 / (N - 1);
    const int j = (int)lround((x + 1.0) / h), i = (int)lround((y + 1.0) / h), k = (int)lround((z + 1.0) / h);
    return (k * N + i) * N + j;
}

In [ ]:
%%writefile fmm3d.h
#pragma once           // il file viene incluso una sola volta
#include <cstdio>    
#include <vector>
#include <cmath>      // fminf, sqrtf
#include <queue>       // std::priority_queue
#include <functional>  // std::greater
#include <utility>     // std::pair
#include <chrono>      // misura dei tempi
#include <fstream>     // scrittura su file
const float INF = 1e30f;                 // "infinito": nodo non ancora raggiunto dall'onda
typedef std::pair<float, int> Entry;     // coppia (u, indice del nodo) per la coda di priorita' del FMM

// Risolutore locale 3D (Godunov, Alg. 2 esteso al 3D): dai minimi dei vicini a, b, c al nuovo u
// Prova con 1, 2, poi 3 vicini; usato sia dal FMM (CPU) sia dal Block FIM (GPU)
__host__ __device__ float solve_quadratic3(float a, float b, float c,float F, float h)
{
    float Fh = F * h;                              // tempo per attraversare una cella
    if (a > b) { float t = a; a = b; b = t; }      // ordina: a <= b <= c
    if (b > c) { float t = b; b = c; c = t; }
    if (a > b) { float t = a; a = b; b = t; }

    float u = a + Fh;                              // 1 vicino (caso 1)
    if (u <= b) return u;
    u = 0.5f * (a + b + sqrtf(2.0f * Fh * Fh - (a - b) * (a - b)));   // 2 vicini (caso 2 = Alg. 2)
    if (u <= c) return u;
    float d = 3.0f * Fh * Fh - ((a - b) * (a - b) + (b - c) * (b - c) + (c - a) * (c - a));
    return (a + b + c + sqrtf(d)) / 3.0f;          // 3 vicini (caso 3)                                  
}

// Nuovo u del nodo (k, i, j) per il FMM: minimo dei vicini per asse + risolutore locale
float solve_node(const std::vector<float>& u, const std::vector<float>& F,
                 int k, int i, int j, int N, float h)
{
    int idx = (k * N + i) * N + j;  // indice globale del nodo (k, i, j)
    // a: min dei vicini lungo x (stessa riga, colonna j-1 e j+1)
    float left  = (j > 0)     ? u[idx - 1] : INF;
    float right = (j < N - 1) ? u[idx + 1] : INF;
    // b: min dei vicini lungo y (stessa colonna, riga i-1 e i+1)
    float down  = (i > 0)     ? u[idx - N] : INF;
    float up    = (i < N - 1) ? u[idx + N] : INF;
    // c: min dei vicini lungo z (stessa colonna e riga, piano k-1 e k+1)
    float back  = (k > 0)     ? u[(idx - N * N)] : INF;
    float front = (k < N - 1) ? u[(idx + N *N)] : INF;

    float a = fminf(left, right);
    float b = fminf(down, up);
    float c = fminf(back, front);

    float q = solve_quadratic3(a, b, c, F[idx], h);
    return fminf(u[idx], q);                                  // return min(u_ijk, u)
}
// Fast Marching Method (Alg. 3) su CPU: il riferimento per il Block FIM
// Fissa i nodi in ordine di tempo crescente con una coda di priorita'
void fmm(std::vector<float>& u, const std::vector<float>& F, int N, float h, const std::vector<int>& srcs, const std::vector<unsigned char>& mask)
{
    enum { LONTANO = 0, ATTIVO = 1, NOTO = 2 };      // stato di ogni nodo
    std::vector<char> state(N * N * N, LONTANO);    // array di char NxNxN, inizializzato a LONTANO

    std::priority_queue<Entry, std::vector<Entry>, std::greater<Entry>> heap;
    for (int s : srcs){heap.push(Entry(u[s], s)); state[s] = ATTIVO;}

    const int dk[6] = {  0, 0,  0, 0, -1, 1 };       // spostamenti verso i 6 vicini:
    const int di[6] = {  0, 0, -1, 1,  0, 0 };       // sinistra, destra, sotto, sopra,
    const int dj[6] = { -1, 1,  0, 0,  0, 0 };       // davanti, dietro

    while (!heap.empty()) {                          // while A != vuoto
        Entry trial = heap.top();                    // Trial <- radice del min-heap
        heap.pop();                                  // Rimuovi Trial da A
        int x = trial.second;                        // indice del nodo Trial
        if (state[x] == NOTO || trial.first > u[x])  // coppia "scaduta": si ignora
            continue;
        state[x] = NOTO;                             // Aggiungi Trial a Set

        int k = x / (N*N), i = (x / N) % N, j = x % N;                    // strato, riga, colonna del nodo Trial
        for (int d = 0; d < 6; ++d) {                // foreach x_nb adiacente a Trial
            int ni = i + di[d], nj = j + dj[d], nk = k + dk[d];
            if (ni < 0 || ni >= N || nj < 0 || nj >= N || nk < 0 || nk >= N) continue;   // fuori dal dominio
            int nb = (nk * N + ni) * N + nj;   // indice del nodo vicino
            if (state[nb] == NOTO || !mask[nb]) continue;         // if x_nb non appartiene a Set

            float q = solve_node(u, F, nk, ni, nj, N, h);
            if (q < u[nb]) {                         // u(x_nb) <- soluzione di g(x_nb)
                u[nb] = q;
                state[nb] = ATTIVO;                  // Aggiungi x_nb ad A
                heap.push(Entry(q, nb));
            }
        }
    }
}

In [ ]:
%%writefile node_fim.cuh
#pragma once           // il file viene incluso una sola volta
// FIM a nodi (Alg. 6 della tesi) sulla GPU: lista attiva di nodi, un thread per nodo, niente shared memory
// Versione con maschera (ostacoli fermi a INF) e piu' sorgenti (da BlockFIM_vs_FIM_FM2_3D.ipynb)
// Si include in fim3d.cu dopo la definizione di CUDA_CHECK; INF e solve_quadratic3 vengono da fmm3d.h
#include <vector>
#include <utility>         // std::swap
#include <cuda_runtime.h>
#include "fmm3d.h"         // INF e solve_quadratic3

// Nuovo u del nodo x, leggendo i 6 vicini dalla memoria globale (INF fuori dalla griglia)
__device__ float solve_global(const float* u, const float* F, int x, int N, float h)
{
    int k = x / (N * N), i = (x / N) % N, j = x % N;             // strato, riga e colonna del nodo
    float a = fminf(j > 0 ? u[x - 1] : INF, j < N - 1 ? u[x + 1] : INF);              // vicini lungo x
    float b = fminf(i > 0 ? u[x - N] : INF, i < N - 1 ? u[x + N] : INF);              // vicini lungo y
    float c = fminf(k > 0 ? u[x - N * N] : INF, k < N - 1 ? u[x + N * N] : INF);      // vicini lungo z
    return fminf(u[x], solve_quadratic3(a, b, c, F[x], h));      // return min(u_x, u)
}

// Alg. 6, prima parte: ogni nodo x della lista A si aggiorna (p <- u(x), q <- g(x), u(x) <- q)
__global__ void node_update(const int* list, int count, float* u, const float* F,
                            int N, float h, float eps, unsigned char* conv)
{
    int t = blockIdx.x * blockDim.x + threadIdx.x;               // un thread per nodo della lista
    if (t >= count) return;
    int x = list[t];
    float p = u[x];
    float q = solve_global(u, F, x, N, h);
    u[x] = q;
    conv[t] = fabsf(p - q) < eps;                                // |p - q| < eps: nodo convergente
}

// Alg. 6, seconda parte: un nodo non convergente resta in A; uno convergente esce
// e prova i vicini fuori da A: se un vicino migliora, entra nella nuova lista
__global__ void node_neighbors(const int* list, int count, const unsigned char* conv,
                               const unsigned char* in_list, const unsigned char* mask, float* u, const float* F, int N, float h,
                               int* next_flag, int* next_list, int* next_count)
{
    int t = blockIdx.x * blockDim.x + threadIdx.x;
    if (t >= count) return;
    int x = list[t];
    if (!conv[t]) {                                              // non convergente: resta attivo
        if (atomicExch(&next_flag[x], 1) == 0) next_list[atomicAdd(next_count, 1)] = x;
        return;
    }
    int k = x / (N * N), i = (x / N) % N, j = x % N;
    int  nb[6] = { x - 1,  x + 1,     x - N, x + N,     x - N * N, x + N * N };   // sinistra, destra, sotto, sopra, davanti, dietro
    bool ok[6] = { j > 0, j < N - 1, i > 0, i < N - 1, k > 0,     k < N - 1 };
    for (int d = 0; d < 6; ++d) {
        if (!ok[d] || in_list[nb[d]] || !mask[nb[d]]) continue;  // fuori griglia, gia' in A, oppure ostacolo
        int y = nb[d];
        float p = u[y];
        float q = solve_global(u, F, y, N, h);
        if (q < p) {                                             // p > q: il vicino migliora
            u[y] = q;
            if (atomicExch(&next_flag[y], 1) == 0)               // aggiungi a A (una volta sola)
                next_list[atomicAdd(next_count, 1)] = y;
        }
    }
}

// Aggiornamento dei flag di appartenenza: la lista vecchia esce
__global__ void clear_in_list(const int* list, int count, unsigned char* in_list)
{
    int t = blockIdx.x * blockDim.x + threadIdx.x;
    if (t < count) in_list[list[t]] = 0;
}

// e la nuova entra (next_flag torna a 0 per l'iterazione successiva)
__global__ void set_in_list(const int* list, int count, unsigned char* in_list, int* next_flag)
{
    int t = blockIdx.x * blockDim.x + threadIdx.x;
    if (t < count) { in_list[list[t]] = 1; next_flag[list[t]] = 0; }
}

// FIM a nodi (Alg. 6) sulla GPU: restituisce u, il tempo del ciclo (ms) e il numero di iterazioni
int node_fim(std::vector<float>& u, const std::vector<float>& F, int N, float h,
             const std::vector<int>& srcs, const std::vector<unsigned char>& mask, float eps, float* ms)
{
    const int n = N * N * N;                                     // numero di nodi
    const size_t bytes = (size_t)n * sizeof(float);
    const int threads = 256;

    // Memoria sulla GPU
    float *d_u, *d_F; int *d_list, *d_next, *d_count, *d_flag; unsigned char *d_conv, *d_in, *d_mask;
    CUDA_CHECK(cudaMalloc(&d_u, bytes));
    CUDA_CHECK(cudaMalloc(&d_F, bytes));
    CUDA_CHECK(cudaMalloc(&d_list, n * sizeof(int)));            // lista A attuale
    CUDA_CHECK(cudaMalloc(&d_next, n * sizeof(int)));            // lista A del passo successivo
    CUDA_CHECK(cudaMalloc(&d_count, sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_flag, n * sizeof(int)));            // 1 = gia' messo nella nuova lista
    CUDA_CHECK(cudaMalloc(&d_conv, n));                          // convergenza dei nodi della lista
    CUDA_CHECK(cudaMalloc(&d_in, n));                            // 1 = nodo in A
    CUDA_CHECK(cudaMalloc(&d_mask, n));                          // 1 = libero, 0 = ostacolo
    CUDA_CHECK(cudaMemcpy(d_u, u.data(), bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_F, F.data(), bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_mask, mask.data(), n, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemset(d_flag, 0, n * sizeof(int)));
    CUDA_CHECK(cudaMemset(d_in, 0, n));

    // All'inizio A contiene le sorgenti
    int count = (int)srcs.size();
    CUDA_CHECK(cudaMemcpy(d_list, srcs.data(), count * sizeof(int), cudaMemcpyHostToDevice));
    set_in_list<<<(count + threads - 1) / threads, threads>>>(d_list, count, d_in, d_flag);
    CUDA_CHECK(cudaGetLastError());

    cudaEvent_t start, stop;
    CUDA_CHECK(cudaEventCreate(&start));
    CUDA_CHECK(cudaEventCreate(&stop));
    CUDA_CHECK(cudaEventRecord(start));                          // si misura solo il ciclo
    int iter = 0;
    while (count > 0) {                                          // while A != vuoto
        ++iter;
        const int blocks = (count + threads - 1) / threads;
        node_update<<<blocks, threads>>>(d_list, count, d_u, d_F, N, h, eps, d_conv);
        CUDA_CHECK(cudaGetLastError());
        CUDA_CHECK(cudaMemset(d_count, 0, sizeof(int)));
        node_neighbors<<<blocks, threads>>>(d_list, count, d_conv, d_in, d_mask, d_u, d_F, N, h,
                                            d_flag, d_next, d_count);
        CUDA_CHECK(cudaGetLastError());
        clear_in_list<<<blocks, threads>>>(d_list, count, d_in);
        CUDA_CHECK(cudaGetLastError());
        CUDA_CHECK(cudaMemcpy(&count, d_count, sizeof(int), cudaMemcpyDeviceToHost));   // dimensione della nuova A
        if (count > 0) {
            set_in_list<<<(count + threads - 1) / threads, threads>>>(d_next, count, d_in, d_flag);
            CUDA_CHECK(cudaGetLastError());
        }
        std::swap(d_list, d_next);                               // la nuova lista diventa quella attuale
    }
    CUDA_CHECK(cudaEventRecord(stop));
    CUDA_CHECK(cudaEventSynchronize(stop));
    CUDA_CHECK(cudaEventElapsedTime(ms, start, stop));
    CUDA_CHECK(cudaEventDestroy(start));
    CUDA_CHECK(cudaEventDestroy(stop));

    // Risultato sulla CPU e pulizia
    CUDA_CHECK(cudaMemcpy(u.data(), d_u, bytes, cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaFree(d_u)); CUDA_CHECK(cudaFree(d_F));
    CUDA_CHECK(cudaFree(d_list)); CUDA_CHECK(cudaFree(d_next)); CUDA_CHECK(cudaFree(d_count));
    CUDA_CHECK(cudaFree(d_flag)); CUDA_CHECK(cudaFree(d_conv)); CUDA_CHECK(cudaFree(d_in));
    CUDA_CHECK(cudaFree(d_mask));
    return iter;
}


In [ ]:
%%writefile fim3d.cu
#include <cstdio>          
#include <cstdlib>         
#include <cuda_runtime.h>
#include <algorithm>       // std::min
#include <chrono>          // tempi sulla CPU
#include <fstream>         // salvataggio su file
#include "fmm3d.h"           // Fast Marching
#include "ambiente.h"        // Ambiente

// Controllo degli errori CUDA: se una chiamata fallisce, stampa dove e perche' ed esce
#define CUDA_CHECK(call)                                                     \
    do {                                                                     \
        cudaError_t err = (call);                                            \
        if (err != cudaSuccess) {                                            \
            printf("Errore CUDA %s:%d: %s\n", __FILE__, __LINE__,            \
                   cudaGetErrorString(err));                                 \
            exit(1);                                                         \
        }                                                                    \
    } while (0)

#include "node_fim.cuh"   // FIM a nodi (Alg. 6): usa CUDA_CHECK (qui sopra), INF e solve_quadratic3

#define BLOCK 8  // blocco 8 x 8 x 8 = 512 nodi, un thread per nodo

// ===================== PARAMETRI DELL'ESPERIMENTO =====================
#define USCITA_ANTICIPATA 1   // 1 = il blocco esce dalle iterazioni interne quando nessun nodo cambia; 0 = sempre NITER iterazioni
const int NITER = 24;          // iterazioni interne (massime) del Block FIM per ogni lavoro di un blocco
// =====================================================================
// Risolutore locale in shared memory: i 6 vicini del nodo sono nella copia s (blocco + halo)
__device__ float solve_in_block(const float s[BLOCK + 2][BLOCK + 2][BLOCK + 2], int z, int y, int x, float F, float h)
{
    float a = fminf(s[z][y][x - 1], s[z][y][x + 1]);               // a <- min dei vicini lungo x
    float b = fminf(s[z][y - 1][x], s[z][y + 1][x]);               // b <- min dei vicini lungo y
    float c = fminf(s[z - 1][y][x], s[z + 1][y][x]);               // c <- min dei vicini lungo z
    return fminf(s[z][y][x], solve_quadratic3(a, b, c, F, h));      // return min(u_ijk, u)
}

// Step 1: niter iterazioni; Step 2: 1 iterazione di prova.
__device__ void process_block(int block, int niter, float* u, const float* F,
                             int N, int nbx, float h, const unsigned char* mask, float eps, unsigned char* Cb_out)
{
    // Allocazione statica di shared memory
    __shared__ float s_u[BLOCK + 2][BLOCK + 2][BLOCK + 2];              
    __shared__ bool s_Cn[BLOCK * BLOCK * BLOCK];                  // i flag Cn dei 512 nodi
    // posizione del thread nel suo blocco CUDA: ognuno da 0 a 7
    int tx = threadIdx.x, ty = threadIdx.y, tz = threadIdx.z;
    // bj: asse x , bi: asse y e bk: asse z; identifica blocco nella griglia
    int bj = block % nbx, bi = (block / nbx) % nbx, bk = block / (nbx * nbx); 
    // numero del thread dentro il blocco; 
    int tid = (tz*BLOCK + ty) * BLOCK + tx; // 0, ..., 511
    // posizione globale rispetto x,y e z: lungo tutta la griglia
    int i = bi * BLOCK + ty, j = bj * BLOCK + tx, k = bk * BLOCK + tz;
    int idx = (k * N + i) * N + j;  // 0, ..., N^3-1
    bool inside = (i < N && j < N && k < N);                          // l'ultimo blocco puo' sporgere
    bool fixed  = !inside || !mask[idx];             // nodo da non aggiornare: fuori dalla griglia o maschera
    float f = inside ? F[idx] : 1.0f;
    // Global -> Shared
    for (int m = tid; m < (BLOCK + 2) * (BLOCK + 2) * (BLOCK + 2); m += BLOCK * BLOCK * BLOCK) {
        // posizione nella shared memory
        int sz = m / ((BLOCK + 2) * (BLOCK + 2)), sy = (m / (BLOCK + 2)) % (BLOCK + 2), sx = m % (BLOCK + 2);
        // coordinate globali della cella (sz, sy, sx): -1 perché c'è l'halo                          
        int gi = bi * BLOCK + sy - 1, gj = bj * BLOCK + sx - 1, gk = bk * BLOCK + sz - 1;
        
        bool in_domain = (gi >= 0 && gi < N && gj >= 0 && gj < N && gk >= 0 && gk < N);
        s_u[sz][sy][sx] = in_domain ? u[(gk * N + gi) * N + gj] : INF;      // fuori dal dominio: INF
    }
    __syncthreads();
    bool conv = true;
    for (int it = 0; it < niter; ++it){
        // p: valore del nodo all'inizio di questa iterazione
        float p = s_u[tz+1][ty+1][tx+1];
        // q: nuovo valore calcolato in questa iterazione
        float q;
        if (fixed) {q = p;}
        else{
        // tx, ty, tz: 0,...,7; +1: nella shared memory c'è l'halo
        q = solve_in_block(s_u, tz+1, ty + 1, tx + 1, f, h);
        }    
        __syncthreads();
        s_u[tz+1][ty + 1][tx + 1] = q;
        conv = fabsf(p - q) < eps;
#if USCITA_ANTICIPATA
        if (!__syncthreads_or(!conv)) break;   // (B) tutti hanno scritto; se nessun nodo e' cambiato, esci
#else
        __syncthreads();                       // (B) tutti hanno scritto (sempre NITER iterazioni)
#endif
    }
    // Shared -> Global: ogni thread riscrive il proprio nodo
    if (inside) u[idx] = s_u[tz+1][ty + 1][tx + 1];
    // Cn del nodo: vero se non è cambiato nell'ultima iterazione
    s_Cn[tid] = conv;
    __syncthreads();
    for (int reduct = BLOCK * BLOCK * BLOCK / 2; reduct > 0; reduct /= 2) {     // 256, ..., 4, 2, 1
        if (tid < reduct) s_Cn[tid] = s_Cn[tid] && s_Cn[tid + reduct];
        __syncthreads();
    }
    if (tid == 0) Cb_out[block] = s_Cn[0];
}
// Step1: ogni blocco della lista A fa niter iterazioni
__global__ void update_active_blocks(const int* active_list, float* u, const float* F,
                                    int N, int nbx, float h, const unsigned char* mask, int niter,
                                    float eps, unsigned char* Cb_step1)
{
    // blockIdx è il numero del blocco CUDA nel lancio in corso
    process_block(active_list[blockIdx.x], niter, u, F, N, nbx, h, mask, eps, Cb_step1);
    // (N = 33) giro: 1 -> blocchi CUDA lanciati: 1 -> blockIdx.x: 0 -> block = active_list[blockIdx.x]: 62
    // (N = 33) giro: 3 -> blocchi CUDA lanciati: 6 -> blockIdx.x: 0, 1, ..., 5 -> block = active_list[blockIdx.x]: 61, 63, 57, 67, 37, 87
}

// costruisce la lista A dei blocchi attivi (Cb = 0)
__global__ void build_active_list(const unsigned char* Cb, unsigned char* in_list,
                                  int* active_list, int* n_active, int n_blocks) 
{
    // blockIdx: blocco nel lancio in corso; blockDim (monodimensionale CUDA): 256; threadIdx:0,...,255
    int b = blockIdx.x * blockDim.x + threadIdx.x;           // numero del thread(blocco) su tutto il lancio
    // N = 65 -> nbx = 9 -> n_blocks = 729 -> list_blocks = 3 -> thread lanciati = 768 -> thread in eccesso = 39 
    if (b >= n_blocks) return;                               // thread(blocco) in eccesso lanciati da CUDA                                   
    in_list[b] = !Cb[b];                                      // fotografia della lista A di questo giro
    if (!Cb[b]) active_list[atomicAdd(n_active, 1)] = b;               // se NON convergente aggiungi thread(blocco) alla lista A
}
// Step 2: i 6 vicini dei blocchi appena convergenti diventano candidati
__global__ void build_candidate_list(const int* active_list, int n_active, int nbx, const unsigned char* in_list,
                                     const unsigned char* Cb_step1, unsigned char* Cb,
                                     int* is_candidate, int* cand_list, int* cand_count)
{
    int pos = blockIdx.x * blockDim.x + threadIdx.x;
    // se è maggiore o uguale al numero di blocchi attivi fai return
    if (pos >= n_active) return;                       
    int b = active_list[pos];         // un blocco della lista A; N = 33 -> A ={61, 63, ...,87}; active_list[0] =61
    Cb[b] = Cb_step1[b];             // leggi il valore di Cb
    if (!Cb_step1[b]) return;        // proseguono solo i blocchi appena convergenti
    // bj: blocco lungo x; bi: blocco lungo y; bk: blocco lungo z
    int bj = b % nbx, bi = (b / nbx) % nbx, bk = b / (nbx * nbx);
    // sinistra, destra, sotto, sopra e davanti, dietro
    int  nb[6] = { b - 1,  b + 1,       b - nbx, b + nbx,  b - nbx * nbx, b + nbx * nbx};
    // verifico se si trova nella griglia
    bool ok[6] = { bj > 0, bj < nbx - 1, bi > 0,  bi < nbx - 1, bk > 0, bk < nbx - 1 };
    for (int d = 0; d < 6; ++d) {
        if (!ok[d]) continue;                                // scarta blocchi fuori dalla griglia
        // scarta blocchi attivi e non convergenti
        if (in_list[nb[d]] && !Cb_step1[nb[d]]) continue;          // d: indica la direzione; N = 33; nb[d] = [61, 63, 57, 67, 37, 87];      
        if (atomicExch(&is_candidate[nb[d]], 1) == 0)                // primo a segnarlo: niente doppioni
            cand_list[atomicAdd(cand_count, 1)] = nb[d];
    }
}
// Step 2: ogni candidato fa 1 iterazione; se cambia, diventa attivo
__global__ void update_candidate_blocks(const int* cand_list, const int* cand_count, int* is_candidate,
                                        float* u, const float* F, int N, int nbx, float h,
                                        const unsigned char* mask, float eps, unsigned char* Cb)
{
    // scarta blocchi CUDA in eccesso
    if (blockIdx.x >= *cand_count) return;
    int b = cand_list[blockIdx.x];
    // blocchi candidati: niter = 1 -> con la nuova informazione dal blocco convergente, questo vicino cambia?
    process_block(b, 1, u, F, N, nbx, h, mask, eps, Cb);
    if (threadIdx.x == 0 && threadIdx.y == 0 && threadIdx.z == 0) is_candidate[b] = 0;   // pronto per il giro successivo
}
// Ogni giro: lista A -> Step 1 -> Step 2; fine quando A e' vuota
int block_fim(std::vector<float>& u, const std::vector<float>& F, int N, float h,
              const std::vector<int>& srcs, const std::vector<unsigned char>& mask, float eps, int niter, float* ms)
{
    // nbx: numero di blocchi per lato; n_blocks: blocchi totali
    const int nbx = (N + BLOCK - 1) / BLOCK, n_blocks = nbx * nbx * nbx;
    // bytes: dimensione totale griglia 3D
    const size_t bytes = (size_t)N * N * N * sizeof(float);
    // Cb: si inizializza vettore di n_blocks valori a 1
    std::vector<unsigned char> Cb(n_blocks, 1);
    // strato, riga e colonna del nodo sorgente
    for (int s : srcs){
    const int sk = s / (N * N), si = (s / N) % N, sj = s % N;
    // Cb: vale 1 tranne nel blocco della sorgente (0) 
    Cb[((sk / BLOCK) * nbx + si / BLOCK) * nbx + sj / BLOCK] = 0;
    }

    // Memoria sulla GPU e copia dei dati iniziali (CPU -> GPU)
    float *d_u, *d_F; unsigned char *d_Cb, *d_Cb_step1, *d_in_list, *d_mask; int *d_active_list, *d_n_active;
    int *d_is_candidate, *d_cand_list, *d_cand_count;
    CUDA_CHECK(cudaMalloc(&d_u, bytes));
    CUDA_CHECK(cudaMalloc(&d_F, bytes));
    CUDA_CHECK(cudaMalloc(&d_Cb, n_blocks));
    CUDA_CHECK(cudaMalloc(&d_Cb_step1, n_blocks));
    CUDA_CHECK(cudaMalloc(&d_in_list, n_blocks));
    CUDA_CHECK(cudaMalloc(&d_active_list, n_blocks * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_n_active, sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_mask, N*N*N));
    // Memoria per i blocchi candidati
    CUDA_CHECK(cudaMalloc(&d_is_candidate, n_blocks * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_cand_list, n_blocks * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_cand_count, sizeof(int)));
    // si copiano d_u, d_F e d_Cb dalla CPU alla GPU
    CUDA_CHECK(cudaMemcpy(d_u, u.data(), bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_F, F.data(), bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_Cb, Cb.data(), n_blocks, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_mask, mask.data(), N*N*N, cudaMemcpyHostToDevice));
    // si azzerano d_Cb_step1 e d_is_candidate
    CUDA_CHECK(cudaMemset(d_Cb_step1, 0, n_blocks));
    CUDA_CHECK(cudaMemset(d_is_candidate, 0, n_blocks * sizeof(int)));
    // list_blocks: blocchi CUDA da 256 threads per avere un thread per ogni blocco di nodi 
    const int threads = 256, list_blocks = (n_blocks + threads - 1) / threads;
    // si misura l'ora prima del ciclo
    cudaEvent_t start, stop;
    CUDA_CHECK(cudaEventCreate(&start));
    CUDA_CHECK(cudaEventCreate(&stop));
    CUDA_CHECK(cudaEventRecord(start));                     
    int iter = 0;
    while (true) {
        // azzera il numero di blocchi attivi                                           
        int n_active = 0;  // contatore su CPU
        CUDA_CHECK(cudaMemset(d_n_active, 0, sizeof(int)));    // contatore su GPU
        // lancia list_blocks CUDA da 256 threads ciascuno, sufficiente per avere un thread ogni blocco
        build_active_list<<<list_blocks, threads>>>(d_Cb, d_in_list, d_active_list, d_n_active, n_blocks);
        CUDA_CHECK(cudaGetLastError());
        // aggiorna n_active con d_n_active
        CUDA_CHECK(cudaMemcpy(&n_active, d_n_active, sizeof(int), cudaMemcpyDeviceToHost));
        if (n_active == 0) break;  // se n_active == 0, lista A vuota, termina ciclo
        ++iter;                 // conta i giri
        // Step 1: foreach b in A in parallelo; lancia n_active blocchi 8x8x8
        update_active_blocks<<<n_active, dim3(BLOCK, BLOCK, BLOCK)>>>(d_active_list, d_u, d_F, N, nbx, h, d_mask, niter, eps, d_Cb_step1);    // si lanciano n_active blocchi CUDA, ognuno con 512 thread
        CUDA_CHECK(cudaGetLastError());
        // azzeriamo il contatore dei candidati sulla GPU
        CUDA_CHECK(cudaMemset(d_cand_count, 0, sizeof(int)));
        // lancia n_active blocchi CUDA da 256 threads
        build_candidate_list<<<(n_active + threads-1) / threads, threads>>>(d_active_list, n_active, nbx, d_in_list, d_Cb_step1, d_Cb,
                                                                          d_is_candidate, d_cand_list, d_cand_count);
        CUDA_CHECK(cudaGetLastError());
        // ogni blocco ha al più 6 vicini, quindi 6*n_active
        update_candidate_blocks<<<std::min(6 * n_active, n_blocks), dim3(BLOCK, BLOCK, BLOCK)>>>(d_cand_list, d_cand_count, d_is_candidate,
                                                                                    d_u, d_F, N, nbx, h, d_mask, eps, d_Cb);
        CUDA_CHECK(cudaGetLastError());
    }
    CUDA_CHECK(cudaEventRecord(stop));                       // "segna l'ora" dopo il ciclo
    CUDA_CHECK(cudaEventSynchronize(stop));                  // aspetta che la GPU ci arrivi
    CUDA_CHECK(cudaEventElapsedTime(ms, start, stop));       // tempo trascorso [ms]
    CUDA_CHECK(cudaEventDestroy(start));
    CUDA_CHECK(cudaEventDestroy(stop));
    // Risultato sulla CPU (GPU -> CPU) e liberazione della memoria
    CUDA_CHECK(cudaMemcpy(u.data(), d_u, bytes, cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaFree(d_u)); CUDA_CHECK(cudaFree(d_F));CUDA_CHECK(cudaFree(d_Cb));
    CUDA_CHECK(cudaFree(d_Cb_step1)); CUDA_CHECK(cudaFree(d_mask));
    CUDA_CHECK(cudaFree(d_in_list)); CUDA_CHECK(cudaFree(d_active_list)); CUDA_CHECK(cudaFree(d_n_active));
    CUDA_CHECK(cudaFree(d_is_candidate)); CUDA_CHECK(cudaFree(d_cand_list)); CUDA_CHECK(cudaFree(d_cand_count));
    return iter;
}
// Confronto FMM (CPU, Alg. 3), FIM a nodi (GPU, Alg. 6) e Block FIM (GPU, Alg. 7) nel FM2 del drone:
// passata 1 = distanza dagli ostacoli (F = 1), passata 2 = tempo da B con la F del FM2 (la stessa F per tutti, dal FMM)
// Il riferimento dei confronti e' sempre il Block FIM: prima FMM vs Block FIM, poi FIM a nodi vs Block FIM
int main()
{
    cudaDeviceProp prop;
    CUDA_CHECK(cudaGetDeviceProperties(&prop, 0));
    printf("GPU: %s | multiprocessori: %d | max thread per blocco: %d | shared memory per blocco: %zu KB\n",
           prop.name, prop.multiProcessorCount, prop.maxThreadsPerBlock, prop.sharedMemPerBlock / 1024);
    printf("Block FIM: NITER = %d, uscita anticipata = %s\n", NITER, USCITA_ANTICIPATA ? "si" : "no");

    CUDA_CHECK(cudaFree(0));                                 // inizializza la GPU prima delle misure
    const float eps = 1e-6f;                                 // tolleranza di convergenza (uguale per i due metodi GPU)
    const int ripetizioni = 3;                               // ogni tempo = il minimo di 3 misure

    if (!ambiente_valido()) {                                // A o B dentro un ostacolo: il problema non ha senso
        printf("Errore: A o B sono dentro un ostacolo, correggi ambiente.h\n");
        return 1;
    }
    {   // riscaldamento della GPU (non misurato): i primi lanci sono piu' lenti
        const int Nw = 65, nw = Nw * Nw * Nw;
        const std::vector<unsigned char> mw = costruisci_maschera(Nw);
        const std::vector<int> sw = { nodo(Nw, B.x, B.y, B.z) };
        std::vector<float> uw(nw, INF); uw[sw[0]] = 0.0f;
        const std::vector<float> Fw(nw, 1.0f);
        float ms = 0.0f;
        std::vector<float> u1 = uw, u2 = uw;
        block_fim(u1, Fw, Nw, 2.0f / (Nw - 1), sw, mw, eps, NITER, &ms);
        node_fim(u2, Fw, Nw, 2.0f / (Nw - 1), sw, mw, eps, &ms);
    }

    struct Riga { int N, giri, it_fim; float d_fmm, d_fim, uA_fmm, uA_bfim, uA_fim; double t_fmm, t_fim, t_bfim; };
    std::vector<Riga> righe;                                 // i risultati, stampati alla fine in due tabelle
    std::ofstream tab("confronto.txt");                      // una riga per N, per il grafico (Python)
    for (int Nk : { 33, 65, 129, 193, 257 }) {
        const float hk = 2.0f / (Nk - 1);                    // passo della griglia nel cubo [-1, 1]^3
        const int n = Nk * Nk * Nk;                          // numero di nodi
        const std::vector<unsigned char> mask = costruisci_maschera(Nk);    // 1 = libero, 0 = ostacolo
        const std::vector<int> srcs = { nodo(Nk, B.x, B.y, B.z) };          // sorgente della passata 2: l'arrivo B
        const int iA = nodo(Nk, A.x, A.y, A.z);              // nodo della partenza A

        // passata 1: distanza dagli ostacoli (sorgenti = nodi liberi che toccano un ostacolo, F = 1)
        const std::vector<int> bordo = nodi_di_bordo(mask, Nk);
        std::vector<float> d0(n, INF);
        for (int s : bordo) d0[s] = 0.0f;
        const std::vector<float> uno(n, 1.0f);
        std::vector<float> d_fmm, d_fim, d_bfim;
        double t1_fmm = 1e30, t1_fim = 1e30, t1_bfim = 1e30;
        for (int r = 0; r < ripetizioni; ++r) {              // 3 misure, si tiene la piu' veloce
            d_fmm = d0;
            auto t0 = std::chrono::high_resolution_clock::now();
            fmm(d_fmm, uno, Nk, hk, bordo, mask);                                  // CPU: Alg. 3
            auto t1 = std::chrono::high_resolution_clock::now();
            t1_fmm = std::min(t1_fmm, std::chrono::duration<double, std::milli>(t1 - t0).count());
            float ms = 0.0f;
            d_fim = d0;
            node_fim(d_fim, uno, Nk, hk, bordo, mask, eps, &ms);                   // GPU: Alg. 6
            t1_fim = std::min(t1_fim, (double)ms);
            d_bfim = d0;
            block_fim(d_bfim, uno, Nk, hk, bordo, mask, eps, NITER, &ms);         // GPU: Alg. 7
            t1_bfim = std::min(t1_bfim, (double)ms);
        }

        // passata 2: tempo da B con la F del FM2 (stessa F per i tre metodi, ricavata dal FMM di riferimento)
        const std::vector<float> Fk = lentezza_fm2(d_fmm);
        std::vector<float> u0(n, INF);
        for (int s : srcs) u0[s] = 0.0f;
        std::vector<float> u_fmm, u_fim, u_bfim;
        double t2_fmm = 1e30, t2_fim = 1e30, t2_bfim = 1e30;
        int it_fim = 0, giri_bfim = 0;
        for (int r = 0; r < ripetizioni; ++r) {
            u_fmm = u0;
            auto t0 = std::chrono::high_resolution_clock::now();
            fmm(u_fmm, Fk, Nk, hk, srcs, mask);                                    // CPU: Alg. 3
            auto t1 = std::chrono::high_resolution_clock::now();
            t2_fmm = std::min(t2_fmm, std::chrono::duration<double, std::milli>(t1 - t0).count());
            float ms = 0.0f;
            u_fim = u0;
            it_fim = node_fim(u_fim, Fk, Nk, hk, srcs, mask, eps, &ms);            // GPU: Alg. 6
            t2_fim = std::min(t2_fim, (double)ms);
            u_bfim = u0;
            giri_bfim = block_fim(u_bfim, Fk, Nk, hk, srcs, mask, eps, NITER, &ms);   // GPU: Alg. 7
            t2_bfim = std::min(t2_bfim, (double)ms);
        }

        // Controlli, sempre rispetto al Block FIM: max|FMM - Block FIM| e max|FIM - Block FIM| (passate 1 e 2),
        // nessun NaN, ostacoli a INF
        float diff_fmm = 0.0f, diff_fim = 0.0f;
        int n_nan = 0, ost_non_inf = 0;
        for (int q = 0; q < n; ++q) {
            diff_fmm = fmaxf(diff_fmm, fmaxf(fabsf(d_fmm[q] - d_bfim[q]), fabsf(u_fmm[q] - u_bfim[q])));
            diff_fim = fmaxf(diff_fim, fmaxf(fabsf(d_fim[q] - d_bfim[q]), fabsf(u_fim[q] - u_bfim[q])));
            if (std::isnan(u_bfim[q]) || std::isnan(u_fim[q])) ++n_nan;
            if (!mask[q] && (u_bfim[q] < INF || u_fim[q] < INF)) ++ost_non_inf;
        }
        if (n_nan > 0 || ost_non_inf > 0)                    // si stampa solo se qualcosa non va
            printf("ATTENZIONE (N = %d): %d valori NaN, %d nodi ostacolo raggiunti dall'onda\n", Nk, n_nan, ost_non_inf);
        if (u_bfim[iA] >= INF) { printf("N = %d: A NON RAGGIUNGIBILE da B\n", Nk); continue; }

        const double t_fmm = t1_fmm + t2_fmm, t_fim = t1_fim + t2_fim, t_bfim = t1_bfim + t2_bfim;   // FM2 completo
        righe.push_back({ Nk, giri_bfim, it_fim, diff_fmm, diff_fim, u_fmm[iA], u_bfim[iA], u_fim[iA], t_fmm, t_fim, t_bfim });
        // per il grafico: N, giri, iterazioni, tempi delle due passate per i tre metodi
        tab << Nk << " " << giri_bfim << " " << it_fim << " " << t1_fmm << " " << t2_fmm << " "
            << t1_fim << " " << t2_fim << " " << t1_bfim << " " << t2_bfim << "\n";
    }

    // Tabella 1: FMM (CPU) vs Block FIM (GPU)
    printf("\nTabella 1 - FMM vs Block FIM (tempi = passata 1 + passata 2 [ms]; giri della passata 2)\n");
    printf("    N   giri BFIM   max|FMM-BFIM|   u(A) FMM   u(A) BFIM    t FMM [ms]   t BFIM [ms]   speedup\n");
    for (const Riga& r : righe)
        printf("%5d   %9d   %13.2e   %8.4f   %9.4f   %11.1f   %11.2f   %7.1f\n",
               r.N, r.giri, r.d_fmm, r.uA_fmm, r.uA_bfim, r.t_fmm, r.t_bfim, r.t_fmm / r.t_bfim);

    // Tabella 2: FIM a nodi (GPU) vs Block FIM (GPU)
    printf("\nTabella 2 - FIM a nodi vs Block FIM (tempi = passata 1 + passata 2 [ms]; iterazioni e giri della passata 2)\n");
    printf("    N   iter FIM   giri BFIM   max|FIM-BFIM|   u(A) FIM   u(A) BFIM    t FIM [ms]   t BFIM [ms]   speedup\n");
    for (const Riga& r : righe)
        printf("%5d   %8d   %9d   %13.2e   %8.4f   %9.4f   %11.1f   %11.2f   %7.1f\n",
               r.N, r.it_fim, r.giri, r.d_fim, r.uA_fim, r.uA_bfim, r.t_fim, r.t_bfim, r.t_fim / r.t_bfim);
    return 0;
}


In [ ]:
!nvcc -O3 -arch=sm_75 fim3d.cu -o confronto && ./confronto

In [ ]:
# Confronto FMM, FIM a nodi e Block FIM: tempi (passata 1 + 2), speedup e iterazioni al variare di N
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

N, giri, it_fim, t1_fmm, t2_fmm, t1_fim, t2_fim, t1_b, t2_b = np.loadtxt("confronto.txt", unpack=True)
t_fmm, t_fim, t_b = t1_fmm + t2_fmm, t1_fim + t2_fim, t1_b + t2_b
GRIGIO, ARANCIO, BLU = "#52514e", "#eb6834", "#2a78d6"             # FMM, FIM a nodi, Block FIM

fig = make_subplots(rows=1, cols=3, horizontal_spacing=0.08,
                    subplot_titles=("Tempo FM2 [ms], scala log", "Speedup del Block FIM", "Iterazioni globali (passata 2)"))
for y, nome, colore in [(t_fmm, "FMM (CPU)", GRIGIO), (t_fim, "FIM a nodi (GPU)", ARANCIO), (t_b, "Block FIM (GPU)", BLU)]:
    fig.add_trace(go.Scatter(x=N, y=y, mode="lines+markers", name=nome, line=dict(color=colore, width=2)), row=1, col=1)
for y, nome, colore in [(t_fmm / t_b, "vs FMM", GRIGIO), (t_fim / t_b, "vs FIM a nodi", ARANCIO)]:
    fig.add_trace(go.Scatter(x=N, y=y, mode="lines+markers+text", text=[f"{s:.1f}" for s in y], textposition="top center",
                             name="speedup " + nome, line=dict(color=colore, width=2, dash="dot")), row=1, col=2)
fig.add_trace(go.Scatter(x=N, y=it_fim, mode="lines+markers", name="iterazioni FIM a nodi", line=dict(color=ARANCIO, width=2)), row=1, col=3)
fig.add_trace(go.Scatter(x=N, y=giri, mode="lines+markers", name="giri Block FIM", line=dict(color=BLU, width=2)), row=1, col=3)
fig.update_yaxes(type="log", row=1, col=1)
fig.update_xaxes(title_text="N (nodi per lato)", tickvals=N)
fig.update_layout(template="plotly_white", height=460, margin=dict(l=40, r=20, t=60, b=40),
                  legend=dict(orientation="h", x=0.5, xanchor="center", y=1.2))
fig.write_html("confronto_tempi.html", include_plotlyjs="cdn")       # anche come file da scaricare
fig.show()
